# System Design & Real-World Engineering

## Q51. How do you design a scalable web service in Python?
### Typical Stack
| Layer            | Tools                              |
| ---------------- | ---------------------------------- |
| Web Framework    | **FastAPI**, **Flask**, **Django** |
| Data Layer       | PostgreSQL / MongoDB / Redis       |
| Message Queue    | RabbitMQ / Kafka / Celery          |
| Containerization | Docker + Kubernetes                |
| Observability    | Prometheus + Grafana + ELK         |

### Key principles
- Stateless API servers
- Database connection pooling
- Caching layer
- Horizontal scalability (load-balanced containers)

In [1]:
# FastAPI microservice skeleton
from fastapi import FastAPI, Depends
from pydantic import BaseModel

app = FastAPI()

class Item(BaseModel):
    name: str
    price: float

@app.post("/items")
async def create_item(item: Item):
    return {"id": 1, "item": item}

## Q52. How do you handle concurrency and parallelism in backend systems?
### Answer:
| Problem           | Technique            | Library          |
| ----------------- | -------------------- | ---------------- |
| Many I/O tasks    | `asyncio`, `aiohttp` | Async I/O        |
| CPU-bound work    | `multiprocessing`    | True parallelism |
| Distributed tasks | Celery + Redis       | Task queues      |

import asyncio, aiohttp

async def fetch(url):

    async with aiohttp.ClientSession() as s:
        async with s.get(url) as r:
            return await r.text()

async def main():

    urls = ["https://example.com"]*5
    results = await asyncio.gather(*(fetch(u) for u in urls))
    print(len(results))

asyncio.run(main())

## Q53. How would you design a caching system?
### Layers of caching:
- In-process → functools.lru_cache
- Local → File-based / diskcache
- Distributed → Redis / Memcached

### Cache-aside pattern:

def get_user(user_id):
    
    if cached := redis.get(user_id):
        return cached
    data = db.fetch(user_id)
    redis.set(user_id, data)
    return data

## Q54. How do you handle messaging or asynchronous jobs?
### Answer:
Use a message queue for background or decoupled tasks.

Example using Celery:

#### Worker nodes pick up tasks → scalable job processing.

from celery import Celery

app = Celery('tasks', broker='redis://localhost:6379')

@app.task

def send_email(address):

    print(f"Sending email to {address}")

## Q55. Design a real-time log analysis service.
### Architecture
- Log producers → Kafka → Consumer service (Python)
- Consumer writes to Elasticsearch. 
    - Elasticsearch is a distributed search and analytics engine for storing and analyzing large volumes of data in near real-time.
- Kibana dashboards visualize alerts

### Implementation fragment
from kafka import KafkaConsumer

import json

consumer = KafkaConsumer('logs', value_deserializer=lambda m: json.loads(m.decode()))

for msg in consumer:

    if "ERROR" in msg.value['level']:
        alert(msg.value)


## Q56. How do you implement rate limiting?
### Options:
- Token bucket / leaky bucket algorithm
- Redis-based counters with TTL

import time, redis

r = redis.Redis()

def allow_request(user):

    key = f"user:{user}:req"
    if r.incr(key) == 1:
        r.expire(key, 60)
    return int(r.get(key)) <= 10  # 10 req/min


## Q57. How would you design a file upload microservice?
### Answer:
- API receives file → Streams to S3 (not local disk)
- Use presigned URLs for direct client upload
- Background job verifies checksum

import boto3

s3 = boto3.client("s3")

url = s3.generate_presigned_url('put_object', Params={'Bucket':'mybucket', 'Key':'file.txt'}, ExpiresIn=3600)

## Q58. What are key database optimization techniques in Python?
### Answer:
1. Use connection pools (sqlalchemy.create_engine(pool_size=10))
2. Batch inserts instead of one-by-one
3. Add indexes for frequent lookups
4. Use ORM lazy loading carefully
5. Apply caching for heavy reads

## Q60. How do you ensure high availability (HA)?
### Answer:
- Deploy multiple replicas behind a load balancer
- Use health checks + auto-restart (Kubernetes, systemd)
- Store state externally (DB, Redis)
- Use graceful shutdowns for rolling updates